# Lab 18: Build, repeat and break a scientific pipeline

**Goals:** validate scientific tables, run Bash and Python together, retain sample identities and train/test labels, compare repeat runs, and test failure without publishing partial results.

The six samples and six compounds below are **synthetic teaching data**, not experimental measurements. `reads_million` is a sample read count in millions; `mol_weight` is in g/mol; `logS` is a synthetic target in log10(mol/L). The two tables are independent examples, not paired experiments.

Use a Python 3 kernel in Linux/WSL, Colab, or Windows with Git Bash. No network downloads or extra Python packages are needed. Every shell command runs in a temporary workspace. Files persist between `run_bash` calls; shell variables do not. The helper captures output so an expected failure does not interrupt the notebook.

In [ ]:
# Setup: choose a real Bash executable and isolate this lab from your own files.
from pathlib import Path
import csv
import os
import shutil
import subprocess
import sys
import tempfile
try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

candidates = []
if os.name == "nt":
    candidates += [Path(os.environ.get("ProgramFiles", "C:/Program Files")) / "Git/bin/bash.exe",
                   Path(os.environ.get("LOCALAPPDATA", "")) / "Programs/Git/bin/bash.exe"]
else:
    found = shutil.which("bash")
    if found:
        candidates.append(Path(found))
bash = next((str(p) for p in candidates if p.is_file()), None)
if bash is None:
    raise RuntimeError("Use Linux/WSL or Colab, or install Git Bash on Windows, then rerun setup.")
workspace_owner = tempfile.TemporaryDirectory(prefix="mm-bash-lab18-")
workspace = Path(workspace_owner.name)
for name in ["data/raw", "scripts", "results", "logs"]:
    (workspace / name).mkdir(parents=True, exist_ok=True)
lab_env = os.environ.copy()
lab_env["PYTHON"] = Path(sys.executable).as_posix()
if os.name == "nt":
    git_tools = str(Path(bash).parent.parent / "usr/bin")
    lab_env["PATH"] = git_tools + os.pathsep + os.environ.get("PATH", "")

def run_bash(script, *, check=True):
    result = subprocess.run([bash, "--noprofile", "--norc", "-c", script],
                            cwd=workspace, env=lab_env, text=True,
                            capture_output=True, timeout=30)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print("[diagnostics]\n" + result.stderr, end="")
    if check and result.returncode:
        raise RuntimeError(f"Bash command failed with status {result.returncode}")
    return result
print("Isolated workspace ready; Colab:", in_colab)


In [ ]:
# Setup: bundled fixtures; keep the raw copies unchanged.
sample_text = 'sample_id\tcondition\treads_million\tqc_status\nS01\tcontrol\t12.4\tpass\nS02\ttreated\t18.2\tpass\nS03\tcontrol\t4.1\tfail\nS04\ttreated\t22.0\tpass\nS05\tcontrol\t15.7\tpass\nS06\ttreated\t7.3\tfail\n'
compound_text = 'compound_id\tsmiles\tmol_weight\tlogS\tsplit\nC001\tCCO\t46.07\t-0.3\ttrain\nC002\tCC(=O)O\t60.05\t-0.7\ttrain\nC003\tc1ccccc1\t78.11\t-2.1\ttest\nC004\tCC(=O)Oc1ccccc1C(=O)O\t180.16\t-2.3\ttrain\nC005\tCn1c(=O)c2c(ncn2C)n(C)c1=O\t194.19\t-0.1\ttest\nC006\tCCCCCCCC\t114.23\t-4.0\ttrain\n'
(workspace / "data/raw/samples.tsv").write_text(sample_text, encoding="utf8", newline="\n")
(workspace / "data/raw/compounds.tsv").write_text(compound_text, encoding="utf8", newline="\n")
print("Created six synthetic samples and six synthetic compounds.")

## The supplied scripts

The next setup cell writes the exact teaching scripts described in episode 18.2. In the student notebook the scripts remain available; your tasks are to run and verify them, not to retype the implementation. The `qc_samples.sh` interface is `INPUT_TSV OUTPUT_TSV`; `prepare_compounds.py` has the same interface. The runner accepts `INPUT_DIR NEW_OUTPUT_DIR` and refuses an existing destination.

Inspect any script with `print((workspace / "scripts/run_pipeline.sh").read_text())`. The QC cutoff is an arbitrary teaching rule, **not** a recommendation for a sequencing assay.

In [ ]:
#| echo: false
# Setup: standalone copies of the course scripts; no remote source is loaded.
script_sources = {'prepare_compounds.py': '#!/usr/bin/env python3\n"""Validate a synthetic descriptor table and preserve its declared train/test split."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport os\nfrom pathlib import Path\nimport tempfile\n\n\ndef prepare(source: Path, destination: Path) -> None:\n    if source.resolve() == destination.resolve():\n        raise ValueError("input and output must differ")\n    fields = ["compound_id", "smiles", "mol_weight", "logS", "split"]\n    rows = []\n    seen = set()\n    with source.open(newline="", encoding="utf8") as handle:\n        reader = csv.DictReader(handle, delimiter="\\t")\n        if reader.fieldnames != fields:\n            raise ValueError("invalid compound header")\n        for line, row in enumerate(reader, 2):\n            if set(row) != set(fields) or any(not row[k] for k in fields):\n                raise ValueError(f"missing or extra fields on row {line}")\n            if row["compound_id"] in seen:\n                raise ValueError(f"duplicate compound_id on row {line}")\n            seen.add(row["compound_id"])\n            weight, logs = float(row["mol_weight"]), float(row["logS"])\n            if not math.isfinite(weight) or weight <= 0 or not math.isfinite(logs):\n                raise ValueError(f"invalid numeric value on row {line}")\n            if row["split"] not in {"train", "test"}:\n                raise ValueError(f"invalid split on row {line}")\n            rows.append(row)\n    if not rows:\n        raise ValueError("no compound records")\n\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temp = None\n    try:\n        with tempfile.NamedTemporaryFile(\n            mode="w", newline="", encoding="utf8", dir=destination.parent,\n            prefix=f".{destination.name}.", suffix=".tmp", delete=False,\n        ) as handle:\n            temp = Path(handle.name)\n            writer = csv.DictWriter(\n                handle, fieldnames=["compound_id", "mol_weight", "logS", "split"],\n                delimiter="\\t", lineterminator="\\n", extrasaction="ignore",\n            )\n            writer.writeheader()\n            writer.writerows(rows)\n        os.replace(temp, destination)\n    finally:\n        if temp is not None:\n            temp.unlink(missing_ok=True)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("input", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    try:\n        prepare(args.input, args.output)\n    except (OSError, ValueError, csv.Error) as exc:\n        parser.exit(2, f"Compound preparation failed: {exc}\\n")\n    print(f"Wrote {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'qc_samples.sh': '#!/usr/bin/env bash\n# Validate the teaching TSV, then keep passing samples with >=10 million reads.\nset -euo pipefail\nexport LC_ALL=C\n\nif (( $# != 2 )); then\n    printf \'Usage: bash qc_samples.sh INPUT_TSV OUTPUT_TSV\\n\' >&2\n    exit 2\nfi\ninput=$1\noutput=$2\n[[ -f "$input" && -r "$input" ]] || { printf \'Unreadable input: %s\\n\' "$input" >&2; exit 2; }\n[[ ! "$input" -ef "$output" ]] || { printf \'Input and output must differ\\n\' >&2; exit 2; }\n[[ -n "$output" && ! -d "$output" ]] || { printf \'Output must name a file\\n\' >&2; exit 2; }\nmkdir -p -- "$(dirname -- "$output")"\ntemp=$(mktemp "${output}.tmp.XXXXXX")\ntrap \'rm -f -- "$temp"\' EXIT\n\nif ! awk -F \'\\t\' \'\nBEGIN { OFS = "\\t" }\nNR == 1 {\n    if ($0 != "sample_id\\tcondition\\treads_million\\tqc_status") {\n        print "Invalid sample header" > "/dev/stderr"; bad = 1; exit 2\n    }\n    print "sample_id", "condition", "reads_million"; next\n}\n{\n    if (NF != 4 || $1 !~ /^S[0-9]+$/ || seen[$1]++ ||\n        ($2 != "control" && $2 != "treated") ||\n        $3 !~ /^[0-9]+([.][0-9]+)?$/ ||\n        ($4 != "pass" && $4 != "fail")) {\n        print "Invalid sample row " NR > "/dev/stderr"; bad = 1; exit 2\n    }\n    if ($4 == "pass" && $3 + 0 >= 10) print $1, $2, $3\n}\nEND {\n    if (NR < 2 && !bad) {\n        print "No sample records" > "/dev/stderr"; bad = 1\n    }\n    if (bad) exit 2\n}\' "$input" > "$temp"; then\n    printf \'Sample QC failed; output was not replaced\\n\' >&2\n    exit 2\nfi\nmv -- "$temp" "$output"\nprintf \'Wrote %s\\n\' "$output" >&2\n', 'run_pipeline.sh': '#!/usr/bin/env bash\n# Publish a complete, recorded teaching run; never overwrite an existing run.\nset -euo pipefail\nexport LC_ALL=C\n\nif (( $# != 2 )); then\n    printf \'Usage: bash run_pipeline.sh INPUT_DIR NEW_OUTPUT_DIR\\n\' >&2\n    exit 2\nfi\ninput=$(cd -- "$1" && pwd)\noutput=${2%/}\n[[ -n "$output" && ! -e "$output" && ! -L "$output" ]] || {\n    printf \'Choose a new output directory: %s\\n\' "$output" >&2; exit 2;\n}\nscript_dir=$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)\npython=${PYTHON:-python3}\ncommand -v "$python" > /dev/null || { printf \'Python not found: %s\\n\' "$python" >&2; exit 2; }\nparent=$(dirname -- "$output")\nmkdir -p -- "$parent"\nparent=$(cd -- "$parent" && pwd)\noutput="$parent/$(basename -- "$output")"\nstage=$(mktemp -d "$parent/.mm-pipeline.XXXXXX")\nstarted_utc=$(date -u +%Y-%m-%dT%H:%M:%SZ)\ncleanup() { if [[ -d "$stage" ]]; then rm -r -- "$stage"; fi; }\ntrap cleanup EXIT\ntrap \'exit 130\' INT\ntrap \'exit 143\' TERM\n\nprintf \'Starting sample QC\\n\' >&2\nbash "$script_dir/qc_samples.sh" "$input/samples.tsv" "$stage/samples-qc.tsv"\nprintf \'Starting compound preparation\\n\' >&2\n"$python" "$script_dir/prepare_compounds.py" "$input/compounds.tsv" "$stage/compounds-model.tsv"\n(\n    cd -- "$input"\n    sha256sum samples.tsv compounds.tsv\n) > "$stage/input-sha256.txt"\n(\n    cd -- "$script_dir"\n    sha256sum qc_samples.sh prepare_compounds.py run_pipeline.sh\n) > "$stage/script-sha256.txt"\n{\n    printf \'started_utc=%s\\n\' "$started_utc"\n    printf \'input_dir=%s\\n\' "$input"\n    printf \'sample_rule=qc_status:pass,reads_million:>=10\\n\'\n    printf \'bash_version=%s\\n\' "$BASH_VERSION"\n    "$python" --version\n} > "$stage/run-info.txt"\n(\n    cd -- "$stage"\n    sha256sum samples-qc.tsv compounds-model.tsv\n) > "$stage/output-sha256.txt"\nprintf \'complete\\n\' > "$stage/_SUCCESS"\n[[ ! -e "$output" && ! -L "$output" ]] || { printf \'Output appeared during run\\n\' >&2; exit 2; }\nmv -T -- "$stage" "$output"\nprintf \'Complete: %s\\n\' "$output" >&2\n'}
for name, source in script_sources.items():
    (workspace / "scripts" / name).write_text(source, encoding="utf8", newline="\n")
print("Installed teaching scripts:", ", ".join(sorted(script_sources)))

## Task 1 — Preflight

Use `run_bash` to show the first line of both input tables and report Bash and the selected Python version. Set `preflight` to the returned result. Invoke Python as `"$PYTHON"` so the same interpreter as the notebook is used. What does each row in each input represent?

In [ ]:
# Task 1: inspect the two headers and the selected interpreters; assign preflight.
preflight = run_bash('head -n 1 data/raw/samples.tsv; head -n 1 data/raw/compounds.tsv; printf "Bash %s\n" "$BASH_VERSION"; "$PYTHON" --version')

In [ ]:
assert preflight.returncode == 0, "Preflight must succeed before processing."
assert "reads_million" in preflight.stdout and "mol_weight" in preflight.stdout
print("Check passed: the expected sample and compound headers are present.")

## Task 2 — Publish the first complete run

Call `bash scripts/run_pipeline.sh data/raw results/run-01` using `run_bash`, assigning its result to `first_run`. Inspect `_SUCCESS`, `samples-qc.tsv` and `compounds-model.tsv` under that run directory. If you rerun this task alone, choose a fresh run name or restart the setup: refusing an existing run is intentional.

In [ ]:
# Task 2: run the complete pipeline once; assign first_run.
first_run = run_bash("bash scripts/run_pipeline.sh data/raw results/run-01")
run_bash("cat results/run-01/_SUCCESS; cat results/run-01/samples-qc.tsv; cat results/run-01/compounds-model.tsv")

In [ ]:
assert first_run.returncode == 0
assert (workspace / "results/run-01/_SUCCESS").read_text().strip() == "complete"
for name in ["input-sha256.txt", "script-sha256.txt", "output-sha256.txt", "run-info.txt"]:
    assert (workspace / "results/run-01" / name).is_file(), f"Missing provenance file: {name}"
print("Check passed: both processing and run recording completed.")

## Task 3 — Check identities, not only counts

Read the original and prepared tables with `csv.DictReader(..., delimiter="\t")`. Assign `retained_ids` to the list of selected sample IDs, and `original_splits` and `prepared_splits` to dictionaries mapping compound ID to split label. Why does preserving the target column not authorise using it as a feature?

In [ ]:
# Task 3: read the output identities and the declared molecular splits.
def read_tsv(relative):
    with (workspace / relative).open(newline="", encoding="utf8") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))
retained_ids = [row["sample_id"] for row in read_tsv("results/run-01/samples-qc.tsv")]
original_splits = {row["compound_id"]: row["split"] for row in read_tsv("data/raw/compounds.tsv")}
prepared_splits = {row["compound_id"]: row["split"] for row in read_tsv("results/run-01/compounds-model.tsv")}
print("Retained samples:", retained_ids)
print("Preserved splits:", prepared_splits)

In [ ]:
assert retained_ids == ["S01", "S02", "S04", "S05"], "QC must keep the expected samples in input order."
assert original_splits == prepared_splits and len(prepared_splits) == 6, "Preserve all compound IDs and their splits."
print("Check passed: sample selection and molecular split identity are correct.")

## Task 4 — Repeat the computation

Run the same pipeline into `results/run-02`. Assign `repeat_equal` to a Boolean that is true only if both data outputs match the first run byte for byte. Use Bash to check `output-sha256.txt` from inside the second run directory; assign the returned result to `hash_check`. Explain why timestamp metadata is not part of the byte equality check.

In [ ]:
# Task 4: run again, compare the data bytes, and verify the output checksum record.
run_bash("bash scripts/run_pipeline.sh data/raw results/run-02")
names = ["samples-qc.tsv", "compounds-model.tsv"]
repeat_equal = all((workspace / "results/run-01" / name).read_bytes() ==
                   (workspace / "results/run-02" / name).read_bytes() for name in names)
hash_check = run_bash("cd results/run-02 && sha256sum -c output-sha256.txt")

In [ ]:
assert repeat_equal, "The deterministic data tables should match across runs."
assert hash_check.returncode == 0 and hash_check.stdout.count(": OK") == 2
print("Check passed: repeat output is identical and both recorded hashes verify.")

## Task 5 — Test the failure path

Copy `data/raw` into a new `data/bad` directory with Python's `shutil.copytree`. Change only the copied sample file: replace `12.4` with `unknown`. Call the runner on `data/bad` with `results/failed-run`, using `check=False` so you can inspect its expected failure. Assign the result to `bad_run`.

Before running, predict whether `results/failed-run/_SUCCESS` will exist. The absence of that marker alone is insufficient: also check that the final run directory was never published.

In [ ]:
# Task 5: damage a copy of one input and capture the expected failure.
shutil.copytree(workspace / "data/raw", workspace / "data/bad")
bad_samples = workspace / "data/bad/samples.tsv"
bad_samples.write_text(sample_text.replace("12.4", "unknown", 1), encoding="utf8", newline="\n")
bad_run = run_bash("bash scripts/run_pipeline.sh data/bad results/failed-run", check=False)
print("Expected nonzero status:", bad_run.returncode)

In [ ]:
assert bad_run.returncode != 0, "Malformed numeric input must fail."
assert not (workspace / "results/failed-run").exists(), "A failed pipeline must not publish its final run."
assert not list((workspace / "results").glob(".mm-pipeline.*")), "Staging directories should be cleaned on this handled failure."
assert (workspace / "data/raw/samples.tsv").read_text() == sample_text, "The raw fixture must remain unchanged."
print("Check passed: no partial run was published and raw data stayed intact.")

## Task 6 — Protect an existing valid file

Read `results/run-01/samples-qc.tsv` as bytes into `before`. Run `qc_samples.sh` with the damaged sample input and that existing good output, capturing the expected failure in `protected_run`. Then read the output into `after`. What would happen if the script redirected directly into the final output instead of using a temporary path?

In [ ]:
# Task 6: check that a rejected replacement leaves the previous result intact.
output = workspace / "results/run-01/samples-qc.tsv"
before = output.read_bytes()
protected_run = run_bash("bash scripts/qc_samples.sh data/bad/samples.tsv results/run-01/samples-qc.tsv", check=False)
after = output.read_bytes()

In [ ]:
assert protected_run.returncode != 0, "Damaged input should be rejected."
assert after == before, "A failed replacement must preserve the previous valid bytes."
assert not list(output.parent.glob("samples-qc.tsv.tmp.*")), "Temporary output should be cleaned."
print("Check passed: failed processing preserved the earlier result.")

## Reflection

1. Four samples passed. Explain why checking their identifiers is stronger than checking only the number four.
2. What did the repeat run establish, and which scientific assumptions did it leave untested?
3. The prepared compound file contains `logS`. Explain how you would prevent target leakage when choosing model features, and why split labels must remain intact.
4. Which input, code and environment records would you add before using this pattern with real sequence alignment or molecular descriptor software?
5. Describe a scheduling policy for runs that take longer than the interval between scheduled starts. How would a failed job reach its owner?

**Optional extension:** copy the QC script, add a validated numeric cutoff argument, record the cutoff, and compare two new named runs. Keep the raw inputs and the original successful run unchanged.